In [1]:
import os
import glob

import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.stats import skew, kurtosis
from scipy.signal import savgol_filter

In [2]:
import sys
sys.path.append("..") 
from pre_processing import extract_all_features, extract_features_from_dataset

In [3]:
BASE_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_0overlap/vitaldb_checkpoints"
UCI_PREPROCESSING_DIR = (
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/"
    "UCI/Feature_extraction/Non_fiducial_features/"
    "8sec_0overlap/data"
)
checkpoint_files = sorted(
    glob.glob(
        os.path.join(
            BASE_DIR,
            "case_*.npz"
        )
    )
)

print(
    "Number of VitalDB case files:",
    len(checkpoint_files)
)

print("\nFirst few files:")
for f in checkpoint_files[:5]:
    print(f)

Number of VitalDB case files: 1992

First few files:
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_0overlap/vitaldb_checkpoints/case_1.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_0overlap/vitaldb_checkpoints/case_10.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_0overlap/vitaldb_checkpoints/case_1001.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_0overlap/vitaldb_checkpoints/case_1002.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_0overlap/vitaldb_checkpoints/case_1003.npz


In [4]:
data = np.load(
    checkpoint_files[0]
)

print("Keys:")
print(data.files)

X = data["X"]
y = data["y"]

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nX dtype:", X.dtype)
print("y dtype:", y.dtype)

Keys:
['X', 'y', 'case_id']

X shape: (1241, 1000)
y shape: (1241, 2)

X dtype: float32
y dtype: float32


In [5]:
features = extract_all_features(X[0])

print("Number of features:", len(features))
print(features)

Number of features: 57
{'ppg_mean': np.float64(603.0905683898926), 'ppg_median': np.float64(596.9935913085938), 'ppg_std': np.float64(35.452832959841395), 'ppg_variance': np.float64(1256.9033648784161), 'ppg_iqr': np.float64(68.47201538085938), 'ppg_skewness': np.float64(0.10077320850827567), 'ppg_kurtosis': np.float64(-0.5435422791802758), 'ppg_zero_crossing_rate': np.float64(0.0), 'ppg_shannon_entropy': np.float64(4.097449048789044), 'ppg_energy_mean': np.float64(364975.1370457221), 'ppg_energy_variance': np.float64(1841768400.9488013), 'ppg_energy_skewness': np.float64(0.21611851193285697), 'ppg_energy_kurtosis': np.float64(-0.946458464687308), 'ppg_energy_iqr': np.float64(82667.95010345802), 'ppg_kte_mean': np.float64(176.71332976427527), 'ppg_kte_variance': np.float64(77092694.82967737), 'ppg_kte_skewness': np.float64(20.416367992419143), 'ppg_kte_kurtosis': np.float64(604.5104272585521), 'ppg_kte_iqr': np.float64(808.8820107635111), 'vpg_mean': np.float64(0.3667749545224035), 'vp

In [ ]:
\

npz_files = sorted(
    glob.glob(
        os.path.join(BASE_DIR, "case_*.npz")
    )
)

print("Number of .npz files:", len(npz_files))


# --------------------------------------------------
# Load each .npz and extract features
# --------------------------------------------------

all_dfs = []

for file in tqdm(npz_files, desc="Extracting features"):

    # Load .npz
    data = np.load(file)

    X = data["X"]
    y = data["y"]

    # Extract 57 features + SBP + DBP
    df_case = extract_features_from_dataset(
        X,
        y,
        fs=125,
        n_jobs=2
    )

    # Add case ID
    case_id = os.path.basename(file).replace(
        ".npz", ""
    )

    df_case["case_id"] = case_id

    all_dfs.append(df_case)


# --------------------------------------------------
# Combine all cases
# --------------------------------------------------

df_vitaldb = pd.concat(
    all_dfs,
    ignore_index=True
)

print("Final shape:", df_vitaldb.shape)
print(df_vitaldb.head())

Number of .npz files: 1992


Extracting features:   2%|▏         | 35/1992 [13:12<11:29:49, 21.15s/it]

In [ ]:
def extract_nonfiducial_features(signal):

    signal = np.asarray(signal, dtype=np.float64)

    signal = signal[np.isfinite(signal)]

    if len(signal) == 0:
        return {}

    # -----------------------------
    # Basic statistical features
    # -----------------------------

    mean_val = np.mean(signal)
    median_val = np.median(signal)
    std_val = np.std(signal)
    variance_val = np.var(signal)

    iqr_val = (
        np.percentile(signal, 75)
        - np.percentile(signal, 25)
    )

    skewness_val = skew(signal)
    kurtosis_val = kurtosis(signal)

    # -----------------------------
    # Zero crossing rate
    # -----------------------------

    zero_crossings = np.sum(
        signal[:-1] * signal[1:] < 0
    )

    zero_crossing_rate = (
        zero_crossings / (len(signal) - 1)
    )

    # -----------------------------
    # Shannon entropy
    # -----------------------------

    hist, _ = np.histogram(
        signal,
        bins=50
    )

    probabilities = hist / np.sum(hist)

    probabilities = probabilities[
        probabilities > 0
    ]

    shannon_entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    # -----------------------------
    # Energy
    # -----------------------------

    energy = signal ** 2

    energy_mean = np.mean(energy)
    energy_variance = np.var(energy)
    energy_skewness = skew(energy)
    energy_kurtosis = kurtosis(energy)

    energy_iqr = (
        np.percentile(energy, 75)
        - np.percentile(energy, 25)
    )

    # -----------------------------
    # Kaiser-Teager Energy
    # -----------------------------

    if len(signal) >= 3:

        kte = (
            signal[1:-1] ** 2
            - signal[:-2] * signal[2:]
        )

        kte_mean = np.mean(kte)
        kte_variance = np.var(kte)
        kte_skewness = skew(kte)
        kte_kurtosis = kurtosis(kte)

        kte_iqr = (
            np.percentile(kte, 75)
            - np.percentile(kte, 25)
        )

    else:

        kte_mean = np.nan
        kte_variance = np.nan
        kte_skewness = np.nan
        kte_kurtosis = np.nan
        kte_iqr = np.nan

    return {

        "mean": mean_val,
        "median": median_val,
        "std": std_val,
        "variance": variance_val,
        "iqr": iqr_val,
        "skewness": skewness_val,
        "kurtosis": kurtosis_val,

        "zero_crossing_rate": zero_crossing_rate,
        "shannon_entropy": shannon_entropy,

        "energy_mean": energy_mean,
        "energy_variance": energy_variance,
        "energy_skewness": energy_skewness,
        "energy_kurtosis": energy_kurtosis,
        "energy_iqr": energy_iqr,

        "kte_mean": kte_mean,
        "kte_variance": kte_variance,
        "kte_skewness": kte_skewness,
        "kte_kurtosis": kte_kurtosis,
        "kte_iqr": kte_iqr
    }


def extract_all_features(ppg, fs=125):

    ppg = np.asarray(ppg, dtype=np.float64)

    # =========================================
    # 1. Original PPG
    # =========================================

    ppg_features = extract_nonfiducial_features(ppg)

    # =========================================
    # 2. VPG - First derivative
    # =========================================

    vpg = np.gradient(ppg)

    # Smooth VPG
    vpg = savgol_filter(
        vpg,
        window_length=11,
        polyorder=3
    )

    # =========================================
    # 3. APG - Second derivative
    # =========================================

    apg = np.gradient(vpg)

    # Smooth APG
    apg = savgol_filter(
        apg,
        window_length=11,
        polyorder=3
    )

    # =========================================
    # 4. Extract features
    # =========================================

    vpg_features = extract_nonfiducial_features(vpg)

    apg_features = extract_nonfiducial_features(apg)

    # =========================================
    # 5. Combine all 57 features
    # =========================================

    features = {}

    for name, value in ppg_features.items():
        features["ppg_" + name] = value

    for name, value in vpg_features.items():
        features["vpg_" + name] = value

    for name, value in apg_features.items():
        features["apg_" + name] = value

    return features


from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd
import numpy as np


def extract_features_single_sample(i, X, y, fs=125):

    # VitalDB X shape = (N, 1000)
    ppg = X[i, :]

    features = extract_all_features(
        ppg,
        fs
    )

    features["SBP"] = y[i, 0]
    features["DBP"] = y[i, 1]

    return features


def extract_features_from_dataset(
    X,
    y,
    fs=125,
    n_jobs=2
):

    results = Parallel(
        n_jobs=n_jobs,
        backend="loky",
        verbose=0
    )(
        delayed(extract_features_single_sample)(
            i, X, y, fs
        )
        for i in range(len(X))
    )

    return pd.DataFrame(results)

In [ ]:
import os
import joblib
import pandas as pd

# --------------------------------------------------
# Directories
# --------------------------------------------------

VITALDB_SAVE_DIR = "./data"
os.makedirs(VITALDB_SAVE_DIR, exist_ok=True)


# --------------------------------------------------
# Function
# --------------------------------------------------

def create_vitaldb_feature_set(method, target):

    name = f"{target}_{method}"

    print("=" * 70)
    print(f"Processing: {name}")
    print("=" * 70)

    # --------------------------------------------------
    # 1. Load UCI preprocessing package
    # --------------------------------------------------

    pkl_path = os.path.join(
        UCI_PREPROCESSING_DIR,
        f"{name}_preprocessing.pkl"
    )

    print("Loading:", pkl_path)

    package = joblib.load(pkl_path)

    selected_features = package["feature_names"]
    scaler = package["scaler"]

    print("Target:", package["target"])
    print("Number of features:", len(selected_features))
    print("Features:")
    print(selected_features)


    # --------------------------------------------------
    # 2. Check VitalDB features
    # --------------------------------------------------

    missing_features = [
        f for f in selected_features
        if f not in df_vitaldb.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing features in df_vitaldb: {missing_features}"
        )


    # --------------------------------------------------
    # 3. Select features from VitalDB
    # --------------------------------------------------

    X_vital = df_vitaldb[selected_features].copy()

    y_vital = df_vitaldb[target].copy()


    # --------------------------------------------------
    # 4. Apply UCI TRAIN-FITTED scaler
    # --------------------------------------------------

    X_vital_scaled = scaler.transform(X_vital)


    # --------------------------------------------------
    # 5. Create DataFrame
    # --------------------------------------------------

    vital_scaled_df = pd.DataFrame(
        X_vital_scaled,
        columns=selected_features,
        index=df_vitaldb.index
    )

    # Add target
    vital_scaled_df[target] = y_vital.values

    # Preserve case ID
    if "case_id" in df_vitaldb.columns:
        vital_scaled_df["case_id"] = df_vitaldb["case_id"].values


    # --------------------------------------------------
    # 6. Save CSV
    # --------------------------------------------------

    output_path = os.path.join(
        VITALDB_SAVE_DIR,
        f"{name}_vitaldb.csv"
    )

    vital_scaled_df.to_csv(
        output_path,
        index=False
    )

    print("\nVitalDB shape:", vital_scaled_df.shape)
    print("Saved:", output_path)
    print()

    return vital_scaled_df

In [ ]:
SBP_FTEST_vitaldb = create_vitaldb_feature_set("FTEST", "SBP")
SBP_MRMR_vitaldb = create_vitaldb_feature_set("MRMR", "SBP")
SBP_RRELIEFF_vitaldb = create_vitaldb_feature_set("RELIEFF", "SBP")

DBP_FTEST_vitaldb = create_vitaldb_feature_set("FTEST", "DBP")
DBP_MRMR_vitaldb = create_vitaldb_feature_set("MRMR", "DBP")
DBP_RRELIEFF_vitaldb = create_vitaldb_feature_set("RELIEFF", "DBP")

Processing: SBP_FTEST
Loading: /data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/8sec_0overlap/data/SBP_FTEST_preprocessing.pkl
Target: SBP
Number of features: 15
Features:
['vpg_skewness', 'apg_zero_crossing_rate', 'apg_skewness', 'ppg_iqr', 'apg_shannon_entropy', 'vpg_shannon_entropy', 'ppg_energy_iqr', 'vpg_energy_skewness', 'ppg_variance', 'vpg_kurtosis', 'vpg_median', 'ppg_energy_variance', 'ppg_std', 'vpg_iqr', 'ppg_kurtosis']

VitalDB shape: (1241, 17)
Saved: ./data/SBP_FTEST_vitaldb.csv

Processing: SBP_MRMR
Loading: /data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/8sec_0overlap/data/SBP_MRMR_preprocessing.pkl
Target: SBP
Number of features: 15
Features:
['apg_energy_variance', 'ppg_zero_crossing_rate', 'ppg_energy_skewness', 'vpg_skewness', 'ppg_energy_variance', 'apg_mean', 'apg_energy_skewness', 'apg_zero_crossing_rate', 'vpg_mean', 'apg_median', 'vpg_median', 'apg_skewness', 'ppg_kte_kurtosis', '

FileNotFoundError: [Errno 2] No such file or directory: '/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/8sec_0overlap/data/SBP_RRELIEFF_preprocessing.pkl'